In [53]:
import os
import glob
import pandas as pd
import numpy as np

In [54]:
exp_root = "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged"

datasets = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
    "MedXpertQA": "/research/d5/gds/yzhong22/datasets/MedXpertQA",
    "OmniMedVQA": "/research/d5/gds/yzhong22/datasets/OmniMedVQA",
}

In [55]:
zs_eval_paths = [
    "vqa/{}/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Instruct",
    "vqa/{}/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-Instruct",
    "vqa/{}/Gemma3/eval_seed0/gemma-3-4b-it",
    "vqa/{}/MedGemma/eval_seed0/medgemma-4b-it",
    "vqa/{}/InternVL3/eval_seed0/InternVL3-8B-hf",
    "vqa-zeroshot/{}/LLaVA-1.5/seed0",
    "vqa-zeroshot/{}/LLaVA-Med/seed0",
    "vqa/{}/gemini-2.5-pro/eval_seed0/none",
    "vqa/{}/o3/eval_seed0/none",
    "vqa/{}/Lingshu/eval_seed0/Lingshu-7B",
]

train_ml_paths = [
    "vqa/{}/Qwen2-VL/eval_seed0/qwen2vl-7b-lora-ML-merged",
    "vqa/{}/Qwen25-VL/eval_seed0/qwen2_5vl-7b-lora-ML-merged",
    "vqa/{}/Gemma3/eval_seed0/gemma-3-4b-it-lora-ML-merged",
    "vqa/{}/MedGemma/eval_seed0/train_lora_ML_seed42-merged",
    # "vqa/{}/Gemma3/eval_seed0/train_lora_ML_seed42-merged",
    "vqa/{}/InternVL3/eval_seed0/intern-vl-8b-lora-ML-merged",
    "vqa/{}/LLaVA-1.5/eval_seed0/train_lora_ML_seed42_llava",
    "vqa/{}/LLaVA-Med/eval_seed0/train__M_seed42_llava_mistral",
    "vqa/{}/Lingshu/eval_seed0/1epoch-lora8",
]

agents = ["mdagent", "ucagent"]
agent_paths = []

for model in zs_eval_paths:
    for agent in agents:
        agent_paths.append(f"{model.replace('vqa-zeroshot', 'vqa')}/{agent}")

general_model_list = ["LLaVA-1.5", "InternVL3",
                      "Gemma3", "Qwen2-VL", "Qwen25-VL"]
medical_model_list = ["LLaVA-Med", "MedGemma", "gemini-2.5-pro", "o3"]

agent_paths

['vqa/{}/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Instruct/mdagent',
 'vqa/{}/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Instruct/ucagent',
 'vqa/{}/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-Instruct/mdagent',
 'vqa/{}/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-Instruct/ucagent',
 'vqa/{}/Gemma3/eval_seed0/gemma-3-4b-it/mdagent',
 'vqa/{}/Gemma3/eval_seed0/gemma-3-4b-it/ucagent',
 'vqa/{}/MedGemma/eval_seed0/medgemma-4b-it/mdagent',
 'vqa/{}/MedGemma/eval_seed0/medgemma-4b-it/ucagent',
 'vqa/{}/InternVL3/eval_seed0/InternVL3-8B-hf/mdagent',
 'vqa/{}/InternVL3/eval_seed0/InternVL3-8B-hf/ucagent',
 'vqa/{}/LLaVA-1.5/seed0/mdagent',
 'vqa/{}/LLaVA-1.5/seed0/ucagent',
 'vqa/{}/LLaVA-Med/seed0/mdagent',
 'vqa/{}/LLaVA-Med/seed0/ucagent',
 'vqa/{}/gemini-2.5-pro/eval_seed0/none/mdagent',
 'vqa/{}/gemini-2.5-pro/eval_seed0/none/ucagent',
 'vqa/{}/o3/eval_seed0/none/mdagent',
 'vqa/{}/o3/eval_seed0/none/ucagent',
 'vqa/{}/Lingshu/eval_seed0/Lingshu-7B/mdagent',
 'vqa/{}/Lingshu/eval_seed0/Lingshu-7B/ucagent']

In [56]:
status_dict = {
    "model": [],
    "task": [],
    "dataset": [],
    "model_type": [],
    "trainable_module": [],
    "path": [],
    "have_eval_result": [],
    "have_prediction": [],
    "have_gpt_score": [],
}

task = "vqa"

for dataset in datasets.keys():
    for path in zs_eval_paths:
        if "LLaVA" in path:
            if dataset in ["MedXpertQA"]:
                continue
            if dataset in ["OmniMedVQA"]:
                path = path.replace("vqa-zeroshot", "vqa")

                if "LLaVA-Med" in path:
                    path = path.replace("seed0", "eval_seed0/llava-med-v1.5-mistral-7b")
                elif "LLaVA-1.5" in path:
                    path = path.replace("seed0", "eval_seed0/llava-v1.5-7b")

        path = path.format(dataset)
        trainable_module = "n/a"
        model = path.split("/")[2]
        if os.path.exists(os.path.join(exp_root, path, "results.csv")):
            have_eval_result = 1
            df = pd.read_csv(os.path.join(exp_root, path, "results.csv"))
            item = df.iloc[0]
            # model = item["model"]
            if not item["model"] == model:
                print(f"Expected {model}, but got {item['model']}")

        else:
            have_eval_result = 0

        try:
            model_type = item["model_type"]
        except:
            if model in general_model_list:
                model_type = "general"
            elif model in medical_model_list:
                model_type = "medical"
            else:
                raise Exception("Unknown model {}".format(model))

        if os.path.exists(os.path.join(exp_root, path, "predictions.json")):
            have_prediction = 1
        else:
            have_prediction = 0

        if os.path.exists(os.path.join(exp_root, path, "gpt_score.csv")):
            have_gpt_score = 1
        else:
            have_gpt_score = 0

        for k in status_dict.keys():
            status_dict[k].append(globals()[k])


for dataset in datasets.keys():
    for path in train_ml_paths:
        if "LLaVA" in path and dataset in ["MedXpertQA"]:
            continue
        path = path.format(dataset)
        trainable_module = "ML"
        model = path.split("/")[2]
        if os.path.exists(os.path.join(exp_root, path, "results.csv")):
            have_eval_result = 1
            df = pd.read_csv(os.path.join(exp_root, path, "results.csv"))
            item = df.iloc[0]
            # model = item["model"]
            if not item["model"] == model:
                print(f"Expected {model}, but got {item['model']}")

        else:
            have_eval_result = 0

        try:
            model_type = item["model_type"]
        except:
            if model in general_model_list:
                model_type = "general"
            elif model in medical_model_list:
                model_type = "medical"
            else:
                raise Exception("Unknown model {}".format(model))

        if os.path.exists(os.path.join(exp_root, path, "predictions.json")):
            have_prediction = 1
        else:
            have_prediction = 0

        if os.path.exists(os.path.join(exp_root, path, "gpt_score.csv")):
            have_gpt_score = 1
        else:
            have_gpt_score = 0

        for k in status_dict.keys():
            status_dict[k].append(globals()[k])

for dataset in datasets.keys():
    for path in agent_paths:
        if "LLaVA" in path and dataset in ["MedXpertQA"]:
            continue
        path = path.format(dataset)
        trainable_module = path.split("/")[-1]
        model = path.split("/")[2]
        if os.path.exists(os.path.join(exp_root, path, "results.csv")):
            have_eval_result = 1
            df = pd.read_csv(os.path.join(exp_root, path, "results.csv"))
            item = df.iloc[0]
            # model = item["model"]
            if not item["model"] == model:
                print(f"Expected {model}, but got {item['model']}")
        else:
            have_eval_result = 0

        try:
            model_type = item["model_type"]
        except:
            if model in general_model_list:
                model_type = "general"
            elif model in medical_model_list:
                model_type = "medical"
            else:
                raise Exception("Unknown model {}".format(model))

        if os.path.exists(os.path.join(exp_root, path, "predictions.json")):
            have_prediction = 1
        else:
            have_prediction = 0

        if os.path.exists(os.path.join(exp_root, path, "gpt_score.csv")):
            have_gpt_score = 1
        else:
            have_gpt_score = 0

        for k in status_dict.keys():
            status_dict[k].append(globals()[k])


df_status = pd.DataFrame.from_dict(status_dict)

df_status

Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected gemini-2.5-pro, but got o3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected gemini-2.5-pro, but got o3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected gemini-2.5-pro, but got o3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected gemini-2.5-pro, but got o3


Expected Qwen25-VL, but got Qwen2.5-VL
Expected Qwen25-VL, but got Qwen2.5-VL
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected InternVL3, but got Gemma3
Expected Qwen25-VL, but got Qwen2.5-VL
Expected Qwen25-VL, but got Qwen2.5-VL
Expected Qwen2-VL, but got MDAgent(Qwen2-VL)
Expected Qwen2-VL, but got UCAgent(Qwen2-VL)
Expected Qwen25-VL, but got MDAgent(Qwen2.5-VL)
Expected Gemma3, but got MDAgent(Gemma3)
Expected MedGemma, but got MDAgent(MedGemma)
Expected Lingshu, but got MDAgent(Lingshu)
Expected Qwen2-VL, but got MDAgent(Qwen2-VL)
Expected Qwen25-VL, but got MDAgent(Qwen2.5-VL)
Expected Gemma3, but got MDAgent(Gemma3)
Expected MedGemma, but got MDAgent(MedGemma)
Expected Lingshu, but got MDAgent(Lingshu)
Expected Qwen2-VL, but got MDAgent(Qwen2-VL)
Expected Qwen2

,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score
0,Qwen2-VL,vqa,SLAKE,general,n/a,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,1
1,Qwen25-VL,vqa,SLAKE,general,n/a,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,1
2,Gemma3,vqa,SLAKE,general,n/a,vqa/SLAKE/Gemma3/eval_seed0/gemma-3-4b-it,1,1,1
3,MedGemma,vqa,SLAKE,medical,n/a,vqa/SLAKE/MedGemma/eval_seed0/medgemma-4b-it,1,1,1
4,InternVL3,vqa,SLAKE,general,n/a,vqa/SLAKE/InternVL3/eval_seed0/InternVL3-8B-hf,1,1,1
...,...,...,...,...,...,...,...,...,...
215,gemini-2.5-pro,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/gemini-2.5-pro/eval_seed0/none/...,0,0,0
216,o3,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/o3/eval_seed0/none/mdagent,0,0,0
217,o3,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/o3/eval_seed0/none/ucagent,0,0,0
218,Lingshu,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/Lingshu/eval_seed0/Lingshu-7B/m...,1,1,0


In [57]:
df_status.to_csv(os.path.join(exp_root, "exp_status_yuan.csv"), index=False)

## combine index with ruinan

In [58]:
df_status["model"].value_counts()

model
Qwen2-VL          24
Qwen25-VL         24
Gemma3            24
MedGemma          24
InternVL3         24
Lingshu           24
LLaVA-1.5         20
LLaVA-Med         20
gemini-2.5-pro    18
o3                18
Name: count, dtype: int64

In [59]:
famalies = []

family_dict = {
    "LLaVA": ["LLaVA-1.5", "LLaVA-Med"],
    "VILA": ["VILA", "VILA-M3"],
    "Gemma": ["Gemma3", "MedGemma"],
    "Qwen": ["Qwen2-VL", "Qwen25-VL", "Lingshu"],
    "Intern": ["InternVL3"],
    "o-family": ["o3"],
    "Gemini": ["gemini-2.5-pro"],
}

for m_ in df_status["model"].tolist():
    for k, v in family_dict.items():
        if m_ in v:
            famalies.append(k)

df_status["model_family"] = famalies

In [60]:
for i in range(len(df_status)):
    item = df_status.iloc[i]

    if item["have_eval_result"]:
        assert os.path.exists(os.path.join(
            exp_root, item["path"])), "{} does not exists".format(item["path"])

    if item["have_gpt_score"]:
        assert os.path.exists(
            os.path.join(exp_root, item["path"], "deekseep_review.json")
        ), "{} does not exists deepseek review".format(item["path"])
        assert os.path.exists(
            os.path.join(exp_root, item["path"], "gpt_score.csv")
        ), "{} does not exists gpt_score.csv".format(item["path"])

In [61]:
# check validality of prediction files

import json

datasets_expect_closed_test_num = {
    "SLAKE": 416,
    "PathVQA": 3362,
    "VQA-RAD": 251,
    "Harvard-FairVLMed10k": 1996,
    "MedXpertQA": 400,
    "OmniMedVQA": 6186,
}

if_finished = []
error_messages = []

for i in range(len(df_status)):
    item = df_status.iloc[i]

    if not item["have_prediction"]:
        finished = 0
        error_message = np.nan
    else:
        # check if prediction file have enough valid items
        with open(os.path.join(exp_root, item["path"], "predictions.json")) as fp:
            data = json.load(fp)
        dataset = item["dataset"]

        valid_count = 0
        for p in data:
            if p["question_type"] in ["closed", "yes/no", "multi-choice"]:
                if "error" not in p.keys():
                    valid_count += 1

        if valid_count == datasets_expect_closed_test_num[dataset]:
            finished = 1
            error_message = np.nan
        else:
            finished = 0
            error_message = f"Expeted closed question num {datasets_expect_closed_test_num[dataset]}, while got only {valid_count} valid in prediction.json file."

    if_finished.append(finished)
    error_messages.append(error_message)

df_status["if_finished"] = if_finished
df_status["error"] = error_messages

In [62]:
df_status.to_csv(os.path.join(exp_root, "exp_status.csv"), index=False)

In [64]:
unfinished_list = df_status.loc[df_status["if_finished"] == 0]

unfinished_list.to_csv(os.path.join(
    exp_root, "unfinished_list.csv"), index=False)

## copy exp files to ruinan

In [35]:
import shutil

for path, have_eval_result in zip(df_status["path"].tolist(), df_status["have_eval_result"].tolist()):
    if not have_eval_result:
        continue

    target_path = os.path.join(
        exp_root,
        "exp_to_ruinan",
        path,
        #    os.path.join(*path.split("/")[:-1])
    )

    # if not os.path.exists(target_path):
    #     os.makedirs(target_path)
    if not os.path.exists(target_path):
        shutil.copytree(os.path.join(exp_root, path), target_path)